# Movie Rental Data Warehouse

This notebook builds and loads the **Movie Rental Data Warehouse** from the original **Sakila OLTP schema** (`fileDB.sql`).


## 1. Import Required Libraries

In [ ]:
import pandas as pd
import mysql.connector
from sqlalchemy import create_engine, text
from datetime import datetime


## 2. Database Configuration

In [3]:
MYSQL_HOST = "localhost"
MYSQL_USER = "root"
MYSQL_PASSWORD = "12345"
SOURCE_DB = "sakila"
DW_DB = "rental_dw"


## 3. Connect to MySQL Server and Create Data Warehouse Database

In [4]:
server_conn = mysql.connector.connect(
    host=MYSQL_HOST,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD
)
server_cursor = server_conn.cursor()

server_cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DW_DB}")
server_conn.commit()

print(f"Database {DW_DB} is ready.")


Database rental_dw is ready.


## 4. Connect to Source OLTP Database and Target Data Warehouse Database

In [5]:
# Source OLTP connection: Sakila
source_conn = mysql.connector.connect(
    host=MYSQL_HOST,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=SOURCE_DB
)
source_cursor = source_conn.cursor()

# Target Data Warehouse connection
dw_conn = mysql.connector.connect(
    host=MYSQL_HOST,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD,
    database=DW_DB
)
dw_cursor = dw_conn.cursor()

# SQLAlchemy engines for pandas read/write
source_engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}/{SOURCE_DB}"
)

dw_engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}/{DW_DB}"
)

print("Connected to source Sakila and target rental_dw successfully.")


Connected to source Sakila and target rental_dw successfully.


## 5. Create Data Warehouse Tables

In [6]:
create_tables_sql = r"""
SET FOREIGN_KEY_CHECKS = 0;

DROP TABLE IF EXISTS Fact_Payment;
DROP TABLE IF EXISTS Fact_Rental;
DROP TABLE IF EXISTS ETL_Reject_Log;
DROP TABLE IF EXISTS DQ_Rules;
DROP TABLE IF EXISTS Dim_Staff;
DROP TABLE IF EXISTS Dim_Store;
DROP TABLE IF EXISTS Dim_Film;
DROP TABLE IF EXISTS Dim_Customer;
DROP TABLE IF EXISTS Dim_Date;

SET FOREIGN_KEY_CHECKS = 1;

CREATE TABLE Dim_Date (
    date_key INT PRIMARY KEY,
    full_date DATE NOT NULL,
    day_of_week VARCHAR(10),
    day_of_month INT,
    month_number INT,
    month_name VARCHAR(10),
    quarter INT,
    year INT,
    is_weekend BOOLEAN,
    season VARCHAR(10)
);

CREATE TABLE Dim_Customer (
    customer_key INT PRIMARY KEY AUTO_INCREMENT,
    customer_id SMALLINT UNSIGNED NOT NULL,
    full_name VARCHAR(100),
    email VARCHAR(100),
    city VARCHAR(50),
    country VARCHAR(50),
    postal_code VARCHAR(15),
    active_status VARCHAR(10),
    create_date DATE,
    UNIQUE KEY uk_dim_customer_id (customer_id)
);

CREATE TABLE Dim_Film (
    film_key INT PRIMARY KEY AUTO_INCREMENT,
    film_id SMALLINT UNSIGNED NOT NULL,
    title VARCHAR(255),
    description TEXT,
    release_year INT,
    language VARCHAR(50),
    rental_duration INT,
    rental_rate DECIMAL(5,2),
    length_minutes INT,
    rating VARCHAR(10),
    category VARCHAR(50),
    special_features VARCHAR(255),
    UNIQUE KEY uk_dim_film_id (film_id)
);

CREATE TABLE Dim_Store (
    store_key INT PRIMARY KEY AUTO_INCREMENT,
    store_id TINYINT UNSIGNED NOT NULL,
    store_address VARCHAR(100),
    city VARCHAR(50),
    country VARCHAR(50),
    postal_code VARCHAR(15),
    UNIQUE KEY uk_dim_store_id (store_id)
);

CREATE TABLE Dim_Staff (
    staff_key INT PRIMARY KEY AUTO_INCREMENT,
    staff_id TINYINT UNSIGNED NOT NULL,
    full_name VARCHAR(100),
    email VARCHAR(100),
    store_id TINYINT UNSIGNED,
    active_status VARCHAR(10),
    UNIQUE KEY uk_dim_staff_id (staff_id)
);

CREATE TABLE Fact_Rental (
    rental_key INT PRIMARY KEY AUTO_INCREMENT,
    rental_id INT NOT NULL,
    customer_key INT,
    film_key INT,
    store_key INT,
    staff_key INT,
    rental_date_key INT,
    return_date_key INT NULL,
    rental_count INT DEFAULT 1,
    rental_duration_days INT NULL,
    expected_duration_days INT,
    late_return_flag BOOLEAN,
    days_overdue INT,
    UNIQUE KEY uk_fact_rental_id (rental_id),
    FOREIGN KEY (customer_key) REFERENCES Dim_Customer(customer_key),
    FOREIGN KEY (film_key) REFERENCES Dim_Film(film_key),
    FOREIGN KEY (store_key) REFERENCES Dim_Store(store_key),
    FOREIGN KEY (staff_key) REFERENCES Dim_Staff(staff_key),
    FOREIGN KEY (rental_date_key) REFERENCES Dim_Date(date_key),
    FOREIGN KEY (return_date_key) REFERENCES Dim_Date(date_key)
);

CREATE TABLE Fact_Payment (
    payment_key INT PRIMARY KEY AUTO_INCREMENT,
    payment_id SMALLINT UNSIGNED NOT NULL,
    customer_key INT,
    staff_key INT,
    film_key INT,
    store_key INT,
    payment_date_key INT,
    rental_id INT,
    payment_amount DECIMAL(10,2),
    payment_count INT DEFAULT 1,
    UNIQUE KEY uk_fact_payment_id (payment_id),
    FOREIGN KEY (customer_key) REFERENCES Dim_Customer(customer_key),
    FOREIGN KEY (staff_key) REFERENCES Dim_Staff(staff_key),
    FOREIGN KEY (film_key) REFERENCES Dim_Film(film_key),
    FOREIGN KEY (store_key) REFERENCES Dim_Store(store_key),
    FOREIGN KEY (payment_date_key) REFERENCES Dim_Date(date_key)
);

CREATE TABLE DQ_Rules (
    rule_id VARCHAR(10) PRIMARY KEY,
    rule_description VARCHAR(255),
    action_if_violated VARCHAR(255)
);

CREATE TABLE ETL_Reject_Log (
    reject_id INT PRIMARY KEY AUTO_INCREMENT,
    rule_id VARCHAR(10),
    source_table VARCHAR(50),
    source_record_id VARCHAR(50),
    rejection_reason VARCHAR(255),
    rejected_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (rule_id) REFERENCES DQ_Rules(rule_id)
);
"""

for statement in create_tables_sql.split(';'):
    stmt = statement.strip()
    if stmt:
        dw_cursor.execute(stmt)

dw_conn.commit()
print("Data warehouse tables created successfully.")


Data warehouse tables created successfully.


## 6. Load Data Quality Rules

In [7]:
dq_rules_sql = """
INSERT INTO DQ_Rules (rule_id, rule_description, action_if_violated) VALUES
('DQ-01', 'rental.customer_id must exist in the customer table', 'Reject record, log error'),
('DQ-02', 'payment.amount must be greater than 0', 'Reject record, log error'),
('DQ-03', 'return_date must be greater than or equal to rental_date when not NULL', 'Flag as error, set return_date to NULL'),
('DQ-04', 'film.rating must be G, PG, PG-13, R, or NC-17', 'Set to Unknown'),
('DQ-05', 'All FK columns in fact tables must reference existing dimension rows', 'Reject fact record, log orphan key'),
('DQ-06', 'No duplicate payment_id values in Fact_Payment', 'Keep first, discard duplicates')
ON DUPLICATE KEY UPDATE
    rule_description = VALUES(rule_description),
    action_if_violated = VALUES(action_if_violated);
"""

dw_cursor.execute(dq_rules_sql)
dw_conn.commit()
print("Data quality rules loaded.")


Data quality rules loaded.


## 7. Load Dim_Date

In [8]:
date_range_query = """
SELECT
    DATE(MIN(dt)) AS min_date,
    DATE(MAX(dt)) AS max_date
FROM (
    SELECT rental_date AS dt FROM rental
    UNION ALL
    SELECT return_date AS dt FROM rental WHERE return_date IS NOT NULL
    UNION ALL
    SELECT payment_date AS dt FROM payment
) x;
"""

min_max = pd.read_sql(date_range_query, source_engine)
min_date = pd.to_datetime(min_max.loc[0, "min_date"])
max_date = pd.to_datetime(min_max.loc[0, "max_date"])

# Add small buffer around real Sakila data
start_date = min_date - pd.Timedelta(days=1)
end_date = max_date + pd.Timedelta(days=1)

dates = pd.date_range(start=start_date, end=end_date, freq="D")

dim_date = pd.DataFrame({
    "date_key": dates.strftime("%Y%m%d").astype(int),
    "full_date": dates.date,
    "day_of_week": dates.day_name(),
    "day_of_month": dates.day,
    "month_number": dates.month,
    "month_name": dates.month_name(),
    "quarter": dates.quarter,
    "year": dates.year,
    "is_weekend": dates.dayofweek.isin([5, 6]),
})

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    return "Autumn"

dim_date["season"] = dim_date["month_number"].apply(get_season)

dim_date.to_sql("Dim_Date", dw_engine, if_exists="append", index=False)
print(f"Loaded {len(dim_date)} rows into Dim_Date.")


Loaded 269 rows into Dim_Date.


C:\Users\jawad\AppData\Local\Temp\ipykernel_7144\518157589.py:47: UserWarning: The provided table name 'Dim_Date' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  dim_date.to_sql("Dim_Date", dw_engine, if_exists="append", index=False)


## 8. Load Dimension Tables

### 8.1 Load Dim_Customer

In [9]:
dim_customer_sql = """
INSERT INTO Dim_Customer (
    customer_id, full_name, email, city, country, postal_code, active_status, create_date
)
SELECT
    c.customer_id,
    TRIM(CONCAT(c.first_name, ' ', c.last_name)) AS full_name,
    COALESCE(NULLIF(TRIM(c.email), ''), 'Unknown') AS email,
    TRIM(ci.city) AS city,
    TRIM(co.country) AS country,
    COALESCE(NULLIF(TRIM(a.postal_code), ''), 'Unknown') AS postal_code,
    CASE WHEN c.active = 1 THEN 'Active' ELSE 'Inactive' END AS active_status,
    DATE(c.create_date) AS create_date
FROM sakila.customer c
JOIN sakila.address a ON c.address_id = a.address_id
JOIN sakila.city ci ON a.city_id = ci.city_id
JOIN sakila.country co ON ci.country_id = co.country_id;
"""

dw_cursor.execute(dim_customer_sql)
dw_conn.commit()
print("Dim_Customer loaded.")


Dim_Customer loaded.


### 8.2 Load Dim_Film

In [10]:
dim_film_sql = """
INSERT INTO Dim_Film (
    film_id, title, description, release_year, language, rental_duration,
    rental_rate, length_minutes, rating, category, special_features
)
SELECT
    f.film_id,
    f.title,
    TRIM(f.description) AS description,
    f.release_year,
    l.name AS language,
    f.rental_duration,
    f.rental_rate,
    f.length AS length_minutes,
    CASE
        WHEN f.rating IN ('G', 'PG', 'PG-13', 'R', 'NC-17') THEN f.rating
        ELSE 'Unknown'
    END AS rating,
    cat.name AS category,
    f.special_features
FROM sakila.film f
JOIN sakila.language l ON f.language_id = l.language_id
LEFT JOIN (
    SELECT film_id, MIN(category_id) AS category_id
    FROM sakila.film_category
    GROUP BY film_id
) fc ON f.film_id = fc.film_id
LEFT JOIN sakila.category cat ON fc.category_id = cat.category_id;
"""

dw_cursor.execute(dim_film_sql)
dw_conn.commit()
print("Dim_Film loaded.")


Dim_Film loaded.


### 8.3 Load Dim_Store

In [11]:
dim_store_sql = """
INSERT INTO Dim_Store (
    store_id, store_address, city, country, postal_code
)
SELECT
    s.store_id,
    TRIM(CONCAT(a.address, COALESCE(CONCAT(' ', NULLIF(a.address2, '')), ''))) AS store_address,
    ci.city,
    co.country,
    COALESCE(NULLIF(TRIM(a.postal_code), ''), 'Unknown') AS postal_code
FROM sakila.store s
JOIN sakila.address a ON s.address_id = a.address_id
JOIN sakila.city ci ON a.city_id = ci.city_id
JOIN sakila.country co ON ci.country_id = co.country_id;
"""

dw_cursor.execute(dim_store_sql)
dw_conn.commit()
print("Dim_Store loaded.")


Dim_Store loaded.


### 8.4 Load Dim_Staff

In [12]:
dim_staff_sql = """
INSERT INTO Dim_Staff (
    staff_id, full_name, email, store_id, active_status
)
SELECT
    st.staff_id,
    TRIM(CONCAT(st.first_name, ' ', st.last_name)) AS full_name,
    COALESCE(NULLIF(TRIM(st.email), ''), 'Unknown') AS email,
    st.store_id,
    CASE WHEN st.active = 1 THEN 'Active' ELSE 'Inactive' END AS active_status
FROM sakila.staff st;
"""

dw_cursor.execute(dim_staff_sql)
dw_conn.commit()
print("Dim_Staff loaded.")


Dim_Staff loaded.


## 9. Load Fact Tables

### 9.1 Load Fact_Rental

In [13]:
fact_rental_sql = """
INSERT INTO Fact_Rental (
    rental_id,
    customer_key,
    film_key,
    store_key,
    staff_key,
    rental_date_key,
    return_date_key,
    rental_count,
    rental_duration_days,
    expected_duration_days,
    late_return_flag,
    days_overdue
)
SELECT
    r.rental_id,
    dc.customer_key,
    df.film_key,
    ds.store_key,
    dst.staff_key,
    CAST(DATE_FORMAT(r.rental_date, '%Y%m%d') AS UNSIGNED) AS rental_date_key,
    CASE
        WHEN r.return_date IS NULL THEN NULL
        ELSE CAST(DATE_FORMAT(r.return_date, '%Y%m%d') AS UNSIGNED)
    END AS return_date_key,
    1 AS rental_count,
    CASE
        WHEN r.return_date IS NULL THEN NULL
        ELSE DATEDIFF(DATE(r.return_date), DATE(r.rental_date))
    END AS rental_duration_days,
    f.rental_duration AS expected_duration_days,
    CASE
        WHEN r.return_date IS NULL THEN 0
        WHEN DATEDIFF(DATE(r.return_date), DATE(r.rental_date)) > f.rental_duration THEN 1
        ELSE 0
    END AS late_return_flag,
    CASE
        WHEN r.return_date IS NULL THEN 0
        WHEN DATEDIFF(DATE(r.return_date), DATE(r.rental_date)) > f.rental_duration
            THEN DATEDIFF(DATE(r.return_date), DATE(r.rental_date)) - CAST(f.rental_duration AS SIGNED)
        ELSE 0
    END AS days_overdue
FROM sakila.rental r
JOIN sakila.inventory i ON r.inventory_id = i.inventory_id
JOIN sakila.film f ON i.film_id = f.film_id
JOIN Dim_Customer dc ON r.customer_id = dc.customer_id
JOIN Dim_Film df ON i.film_id = df.film_id
JOIN Dim_Store ds ON i.store_id = ds.store_id
JOIN Dim_Staff dst ON r.staff_id = dst.staff_id
WHERE r.rental_date IS NOT NULL;
"""

with dw_engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.execute(text("TRUNCATE TABLE Fact_Rental;"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    conn.execute(text(fact_rental_sql))

print("Fact_Rental loaded successfully")

Fact_Rental loaded successfully


### 9.2 Load Fact_Payment

In [14]:
fact_payment_sql = """
INSERT INTO Fact_Payment (
    payment_id,
    customer_key,
    staff_key,
    film_key,
    store_key,
    payment_date_key,
    rental_id,
    payment_amount,
    payment_count
)
SELECT
    p.payment_id,
    dc.customer_key,
    dst.staff_key,
    df.film_key,
    ds.store_key,
    CAST(DATE_FORMAT(p.payment_date, '%Y%m%d') AS UNSIGNED) AS payment_date_key,
    p.rental_id,
    p.amount AS payment_amount,
    1 AS payment_count
FROM sakila.payment p
JOIN sakila.rental r ON p.rental_id = r.rental_id
JOIN sakila.inventory i ON r.inventory_id = i.inventory_id
JOIN Dim_Customer dc ON p.customer_id = dc.customer_id
JOIN Dim_Staff dst ON p.staff_id = dst.staff_id
JOIN Dim_Film df ON i.film_id = df.film_id
JOIN Dim_Store ds ON i.store_id = ds.store_id
WHERE p.amount > 0
  AND p.payment_date IS NOT NULL;
"""

with dw_engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.execute(text("TRUNCATE TABLE Fact_Payment;"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))
    conn.execute(text(fact_payment_sql))

print("Fact_Payment loaded successfully")

Fact_Payment loaded successfully


## 10. Data Quality Checks

In [15]:
dq_checks = {
    "DQ-01 rental customer orphan check": """
        SELECT COUNT(*) AS issue_count
        FROM sakila.rental r
        LEFT JOIN sakila.customer c ON r.customer_id = c.customer_id
        WHERE c.customer_id IS NULL;
    """,
    "DQ-02 invalid payment amount": """
        SELECT COUNT(*) AS issue_count
        FROM sakila.payment
        WHERE amount <= 0 OR amount IS NULL;
    """,
    "DQ-03 return date before rental date": """
        SELECT COUNT(*) AS issue_count
        FROM sakila.rental
        WHERE return_date IS NOT NULL AND return_date < rental_date;
    """,
    "DQ-04 invalid film rating": """
        SELECT COUNT(*) AS issue_count
        FROM sakila.film
        WHERE rating NOT IN ('G', 'PG', 'PG-13', 'R', 'NC-17') OR rating IS NULL;
    """,
    "DQ-05 orphan fact rental keys": """
        SELECT COUNT(*) AS issue_count
        FROM Fact_Rental fr
        LEFT JOIN Dim_Customer dc ON fr.customer_key = dc.customer_key
        LEFT JOIN Dim_Film df ON fr.film_key = df.film_key
        LEFT JOIN Dim_Store ds ON fr.store_key = ds.store_key
        LEFT JOIN Dim_Staff dst ON fr.staff_key = dst.staff_key
        WHERE dc.customer_key IS NULL
           OR df.film_key IS NULL
           OR ds.store_key IS NULL
           OR dst.staff_key IS NULL;
    """,
    "DQ-06 duplicate payment_id in Fact_Payment": """
        SELECT COUNT(*) AS issue_count
        FROM (
            SELECT payment_id
            FROM Fact_Payment
            GROUP BY payment_id
            HAVING COUNT(*) > 1
        ) x;
    """
}

results = []
for check_name, sql in dq_checks.items():
    df = pd.read_sql(sql, dw_engine)
    results.append({"check": check_name, "issue_count": int(df.loc[0, "issue_count"])})

pd.DataFrame(results)


,check,issue_count
0,DQ-01 rental customer orphan check,0
1,DQ-02 invalid payment amount,24
2,DQ-03 return date before rental date,0
3,DQ-04 invalid film rating,0
4,DQ-05 orphan fact rental keys,0
5,DQ-06 duplicate payment_id in Fact_Payment,0


## 11. Validate Loaded Row Counts

In [16]:
row_count_sql = """
SELECT 'Dim_Date' AS table_name, COUNT(*) AS row_count FROM Dim_Date
UNION ALL SELECT 'Dim_Customer', COUNT(*) FROM Dim_Customer
UNION ALL SELECT 'Dim_Film', COUNT(*) FROM Dim_Film
UNION ALL SELECT 'Dim_Store', COUNT(*) FROM Dim_Store
UNION ALL SELECT 'Dim_Staff', COUNT(*) FROM Dim_Staff
UNION ALL SELECT 'Fact_Rental', COUNT(*) FROM Fact_Rental
UNION ALL SELECT 'Fact_Payment', COUNT(*) FROM Fact_Payment
UNION ALL SELECT 'DQ_Rules', COUNT(*) FROM DQ_Rules;
"""

pd.read_sql(row_count_sql, dw_engine)


,table_name,row_count
0,Dim_Date,269
1,Dim_Customer,599
2,Dim_Film,1000
3,Dim_Store,2
4,Dim_Staff,2
5,Fact_Rental,16044
6,Fact_Payment,16020
7,DQ_Rules,6


## 14. Close Connections

In [19]:
source_cursor.close()
source_conn.close()
dw_cursor.close()
dw_conn.close()
server_cursor.close()
server_conn.close()

print("Connections closed.")


Connections closed.
